# Set Library Path

In [1]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

# Load Libraries

In [3]:
library(tidyverse)
library(WGCNA)
library(magrittr)
library(biomaRt)
library(dplyr)

# Load Metadata

In [4]:
# Load WGCNA metadata
metadata <- read.csv("metadata.csv")

# Set row names
rownames(metadata) <- metadata$Sample_Name
metadata$Sample_Name <- NULL

# Clean up rows from DEG analysis
metadata$Gene <- NULL
metadata$Sample_ID <- NULL
metadata$Cell_Type <- NULL
metadata$Genotype <- NULL
metadata$Time_Point <- NULL
metadata$Group <- NULL
metadata$Replicate <- NULL
metadata$Clean_Sample_Name <- NULL
metadata$Batch <- NULL

# Remove comparisons we are not interested in 
metadata$D18_WT_vs_D46_WT <- NULL
metadata$D46_WT_vs_D90_WT <- NULL
metadata$D18_WT_vs_D90_WT <- NULL
metadata$D18_MUT_vs_D46_MUT <- NULL
metadata$D46_MUT_vs_D90_MUT <- NULL
metadata$D18_MUT_vs_D90_MUT <- NULL


# Sort metadata by row names
metadata <- metadata[order(rownames(metadata)), ]

# View
head(metadata)

,D18_MUT_vs_D18_WT,D46_MUT_vs_D46_WT,D90_MUT_vs_D90_WT,Decreased_GAD1_Expression,Decreased_FOXG1_Expression,Smaller_Organoid_Area,Impaired_Neurite_Outgrowth
,<int>,<int>,<int>,<int>,<int>,<int>,<int>
CO_R270X_B1_d18_A1,1,NA,NA,NA,1,NA,NA
CO_R270X_B1_d18_A2,1,NA,NA,NA,1,NA,NA
CO_R270X_B1_d18_A3,1,NA,NA,NA,1,NA,NA
CO_R270X_B1_d18_D1,0,NA,NA,NA,0,NA,NA
CO_R270X_B1_d18_D2,0,NA,NA,NA,0,NA,NA
CO_R270X_B1_d18_D3,0,NA,NA,NA,0,NA,NA


# Load Data

In [5]:
# Read in data
MEs0 <- read.table("08_WGCNA/MECP2_module_eigengenes.txt", header = TRUE, sep = "\t")

# Covnert to df
MEs0 <- as.data.frame(MEs0)

# Set rownames
rownames(MEs0) <- MEs0$sample_id
MEs0$sample_id <- NULL

# View
head(MEs0)

,MEorangered4,MEsalmon4,MEgreen,MElightsteelblue1,MEsalmon,MEpurple,MEyellow,MEmediumpurple3,MEdarkturquoise,MEmidnightblue,⋯,MEsaddlebrown,MEturquoise,MEdarkolivegreen,MEdarkgreen,MEroyalblue,MEthistle2,MElightcyan,MEskyblue,MEsteelblue,MEgrey
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CO_R270X_B1_d18_A1,0.10383141,0.1406336,0.2325975,0.10768825,-0.01761850,0.13762149,0.1689046,0.13708045,0.03685824,0.07877443,⋯,-0.2477044,-0.2410736,-0.03913242,-0.034556926,-0.07976427,0.044612829,0.2031119,0.11127951,0.1870922,0.2106989
CO_R270X_B1_d18_A2,0.18697548,0.2051027,0.2497342,0.03083450,-0.03856644,0.04013022,0.1655460,-0.05686887,0.03113315,-0.04149564,⋯,-0.2378553,-0.2567479,0.05475004,-0.031195013,-0.10666023,0.145279869,0.2538338,0.16552825,0.2437258,0.3175530
CO_R270X_B1_d18_A3,0.12585440,0.1326752,0.2271875,0.07263541,0.01192403,0.08151507,0.1648110,0.07461271,0.05036924,0.04900288,⋯,-0.2337344,-0.2422608,0.03050992,0.001494034,-0.09314762,0.129194443,0.2200357,0.13070149,0.2001811,0.2434773
CO_R270X_B1_d18_D1,0.04869635,0.1195829,0.2294042,0.12512477,-0.01738667,0.12412002,0.1417504,0.18223864,0.07305108,0.10238053,⋯,-0.2260527,-0.2226575,-0.08330721,-0.046925258,-0.07754900,-0.019516521,0.1973607,0.08464721,0.1974587,0.2073265
CO_R270X_B1_d18_D2,0.13716610,0.1110144,0.2788223,0.14736848,-0.05529843,0.08799831,0.1298296,0.19228776,-0.01770189,0.02109205,⋯,-0.2297787,-0.2342391,0.03990921,0.013600623,-0.08743747,0.104901509,0.2446465,0.13966526,0.2870873,0.2812685
CO_R270X_B1_d18_D3,0.16990097,0.1759647,0.2882205,0.15727401,-0.09249766,0.03836725,0.1189377,0.21192769,-0.02663957,-0.02155713,⋯,-0.1979303,-0.2440408,-0.02864060,0.016464840,-0.07796142,-0.004157998,0.2363792,0.18181795,0.2534982,0.3470861


# Calculate Correlations

In [6]:
# Confirm sample order matches
table(rownames(MEs0) == rownames(metadata)) # All TRUE

# Pearson correlation
moduleTraitCorP <- WGCNA::cor(MEs0, metadata, method = "pearson", use = "p")
moduleTraitPvalueP <- corPvalueFisher(moduleTraitCorP, length(rownames(metadata)), twoSided = TRUE)

# Remove columns without significance
significant_modules <- rownames(moduleTraitPvalueP)[
  apply(moduleTraitPvalueP, 1, function(x) any(x < 0.05, na.rm = TRUE))
]

# Keep only modules with significant correlations
moduleTraitCorP <- moduleTraitCorP[significant_modules, ]
moduleTraitPvalueP <- moduleTraitPvalueP[significant_modules, ]

# Delete grey module
moduleTraitCorP <- moduleTraitCorP[!rownames(moduleTraitCorP) %in% "MEgrey", ]
moduleTraitPvalueP <- moduleTraitPvalueP[!rownames(moduleTraitPvalueP) %in% "MEgrey", ]

head(moduleTraitCorP)
head(moduleTraitPvalueP)


TRUE 
  32 

,D18_MUT_vs_D18_WT,D46_MUT_vs_D46_WT,D90_MUT_vs_D90_WT,Decreased_GAD1_Expression,Decreased_FOXG1_Expression,Smaller_Organoid_Area,Impaired_Neurite_Outgrowth
MEorangered4,0.51202669,-0.37925842,-0.47230845,-0.37075449,-0.13961057,-0.47230845,-0.47230845
MEsalmon4,0.25102531,-0.43469011,-0.12707301,-0.17265973,-0.05568184,-0.12707301,-0.12707301
MElightsteelblue1,-0.75825482,0.09046882,0.05487812,-0.05906646,-0.13884900,0.05487812,0.05487812
MEsalmon,0.25510565,-0.16828318,0.73184672,-0.01395116,0.04564494,0.73184672,0.73184672
MEpurple,-0.01173966,-0.40177278,0.28610209,-0.08855523,-0.04712264,0.28610209,0.28610209
MEyellow,0.89515882,-0.33882629,0.29288791,-0.12110560,-0.02725991,0.29288791,0.29288791


,D18_MUT_vs_D18_WT,D46_MUT_vs_D46_WT,D90_MUT_vs_D90_WT,Decreased_GAD1_Expression,Decreased_FOXG1_Expression,Smaller_Organoid_Area,Impaired_Neurite_Outgrowth
MEorangered4,2.325535e-03,0.03157728,5.730941e-03,0.03604394,0.4491886,5.730941e-03,5.730941e-03
MEsalmon4,1.671771e-01,0.01215256,4.914348e-01,0.34762319,0.7640503,4.914348e-01,4.914348e-01
MElightsteelblue1,9.162147e-08,0.62517785,7.673638e-01,0.75014112,0.4516986,7.673638e-01,7.673638e-01
MEsalmon,1.600783e-01,0.36022542,5.094948e-07,0.94010797,0.8057004,5.094948e-07,5.094948e-07
MEpurple,9.495890e-01,0.02185986,1.129804e-01,0.63255273,0.7995332,1.129804e-01,1.129804e-01
MEyellow,6.493114e-15,0.05747210,1.042001e-01,0.51221645,0.8832619,1.042001e-01,1.042001e-01


# Make Heatmap

In [7]:
# Represent significant P-values with * in heatmap
star <- apply(moduleTraitPvalueP, 2, function(x) {
  sapply(x, function(y) {
    ifelse(y < 0.001, "***", 
           ifelse(y < 0.01, "**", 
                  ifelse(y < 0.05, "*", "")))
  })
})

In [8]:
# Clean up column names
clean_colnames <- colnames(moduleTraitCorP) %>%
  gsub("_CtO_", " ", .) %>% 
  gsub("_", " ", .)

In [12]:
# Initialize PDF
pdf("08_WGCNA/Filtered_Module_Trait_Heatmap.pdf", width = 5.5, height = 7)

# Adjust the graphic window and margin settings
sizeGrWindow(width = 5.5, height = 7)
par(mar = c(8, 6, 2, 2))

# Plot the heatmap
labeledHeatmap(
  Matrix = moduleTraitCorP,
  xLabels = clean_colnames,
  yLabels = rownames(moduleTraitCorP),
  ySymbols = gsub("ME", "", rownames(moduleTraitCorP)),
  colorLabels = FALSE,
  colors = blueWhiteRed(50),
  textMatrix = star,
  setStdMargins = FALSE,
  cex.text = 0.6,
  textAdj = c(0.5, 0.8),
  zlim = c(-1, 1),
  main = "Module Trait Pearson Correlation",
  cex.lab.x = 0.5,
  cex.lab.y = 0.6,
  plotLegend = TRUE,
  legendLabel = "Pearson Correlation Coefficient",
  xLabelsAngle = 90
)

# Close the PDF device
dev.off()

pdf 
  2

In the above figure generate, blue is correlated with whatever we coded as 0 and red is correlated with whatever we coded as 1. Anything with a star means it's a significant correlation